In [3]:
import pandas as pd

news = pd.read_pickle("processed_data/news_clean.pkl")

print("Loaded news:", news.shape)

Loaded news: (51282, 8)


In [1]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer('all-MiniLM-L6-v2')

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [6]:
def clean_text(text):
    if not isinstance(text, str):
        return ""
    return text.strip()

titles = news["title"].apply(clean_text).tolist()
abstracts = news["abstract"].apply(clean_text).tolist()
news_ids = news["news_id"].tolist()

In [7]:
title_embeddings = model.encode(
    titles,
    batch_size=64,
    show_progress_bar=True
)

abstract_embeddings = model.encode(
    abstracts,
    batch_size=64,
    show_progress_bar=True
)

Batches:   0%|          | 0/802 [00:00<?, ?it/s]

Batches:   0%|          | 0/802 [00:00<?, ?it/s]

In [ ]:
import numpy as np

text_embeddings = 0.5 * title_embeddings + 0.5 * abstract_embeddings


print("Final embedding shape:", text_embeddings.shape)

Final embedding shape: (51282, 384)


In [52]:
from sklearn.preprocessing import normalize

text_embeddings = normalize(text_embeddings)

In [53]:
np.save("processed_data/text_embeddings.npy", text_embeddings)

In [54]:
import pickle

news_text_dict = {
    news_id: emb for news_id, emb in zip(news_ids, text_embeddings)
}

with open("processed_data/news_text_dict.pkl", "wb") as f:
    pickle.dump(news_text_dict, f)

In [56]:
import numpy as np
import pickle

emb_loaded = np.load("processed_data/text_embeddings.npy")

with open("processed_data/news_text_dict.pkl", "rb") as f:
    dict_loaded = pickle.load(f)

print("Embedding shape:", emb_loaded.shape)
print("Sample vector:", emb_loaded[0][:5])

Embedding shape: (51282, 384)
Sample vector: [-0.01442579  0.07025318  0.07167203 -0.00167617  0.0551527 ]


# Entity Embedding Integration

In [10]:

news = pd.read_pickle("processed_data/news_clean.pkl")
print("Loaded news:", news.shape)

Loaded news: (51282, 8)


In [18]:
entity_embedding_path = "../data/raw/entity_embedding.vec"


entity_dict = {}

with open(entity_embedding_path, "r", encoding="utf-8") as f:
    first_line = f.readline().strip().split()

    # Check if first line is header (e.g., "100000 100")
    if len(first_line) == 2 and first_line[0].isdigit():
        print("Header detected:", first_line)
    else:
        # If not header, process first line
        parts = first_line
        entity_id = parts[0]
        vector = np.array(parts[1:], dtype=np.float32)
        entity_dict[entity_id] = vector

    # Process remaining lines
    for line in f:
        parts = line.strip().split()
        if len(parts) < 2:
            continue
        
        entity_id = parts[0]
        try:
            vector = np.array(parts[1:], dtype=np.float32)
            entity_dict[entity_id] = vector
        except:
            continue

print("Loaded entity embeddings:", len(entity_dict))

Loaded entity embeddings: 26904


In [19]:
entity_dim = len(next(iter(entity_dict.values())))
print("Entity embedding dim:", entity_dim)

Entity embedding dim: 100


In [20]:
def extract_entity_ids(row):
    entity_ids = []

    for ent in row["title_entities"]:
        if "WikidataId" in ent:
            entity_ids.append(ent["WikidataId"])

    for ent in row["abstract_entities"]:
        if "WikidataId" in ent:
            entity_ids.append(ent["WikidataId"])

    return entity_ids

In [21]:
def get_entity_embedding(entity_ids):
    vectors = []

    for eid in entity_ids:
        if eid in entity_dict:
            vectors.append(entity_dict[eid])

    if len(vectors) == 0:
        return np.zeros(entity_dim)

    return np.mean(vectors, axis=0)

In [22]:
# Build Entity Embedding for All News
from tqdm import tqdm
import numpy as np

entity_features = []

for _, row in tqdm(news.iterrows(), total=len(news)):
    entity_ids = extract_entity_ids(row)
    emb = get_entity_embedding(entity_ids)
    entity_features.append(emb)

entity_features = np.array(entity_features)

print("Entity feature shape:", entity_features.shape)

100%|██████████| 51282/51282 [00:00<00:00, 61327.71it/s]

Entity feature shape: (51282, 100)


In [23]:
entity_features = normalize(entity_features)

In [24]:
np.save("processed_data/entity_embeddings.npy", entity_features)

In [25]:
import pickle

news_ids = news["news_id"].tolist()

news_entity_dict = {
    news_id: emb for news_id, emb in zip(news_ids, entity_features)
}

with open("processed_data/news_entity_dict.pkl", "wb") as f:
    pickle.dump(news_entity_dict, f)

In [26]:
print(entity_features.shape)
print(entity_features[0][:5])

(51282, 100)
[ 0.00812286 -0.07991525 -0.01676491  0.15844444 -0.04050017]


# Category Embedding


In [8]:
news = pd.read_pickle("processed_data/news_clean.pkl")

# model = SentenceTransformer('all-MiniLM-L6-v2')

In [9]:
def clean_text(text):
    if not isinstance(text, str):
        return "unknown"
    return text.strip().lower()

categories = news["category"].apply(clean_text).tolist()
subcategories = news["subcategory"].apply(clean_text).tolist()
news_ids = news["news_id"].tolist()

In [10]:
category_embeddings = model.encode(
    categories,
    batch_size=64,
    show_progress_bar=True
)

subcategory_embeddings = model.encode(
    subcategories,
    batch_size=64,
    show_progress_bar=True
)

Batches:   0%|          | 0/802 [00:00<?, ?it/s]

Batches:   0%|          | 0/802 [00:00<?, ?it/s]

In [61]:
# category_features = np.concatenate(
#     [category_embeddings, subcategory_embeddings],
#     axis=1
# )
category_features  = 0.6 * category_embeddings + 0.4 * subcategory_embeddings

print("Category feature shape:", category_features.shape)

Category feature shape: (51282, 384)


In [62]:
category_features = normalize(category_features)

In [63]:
np.save("processed_data/category_embeddings.npy", category_features)

In [64]:
import pickle

news_category_dict = {
    news_id: emb for news_id, emb in zip(news_ids, category_features)
}

with open("processed_data/news_category_dict.pkl", "wb") as f:
    pickle.dump(news_category_dict, f)

In [34]:
print(category_features.shape)
print(category_features[0][:5])

(51282, 768)
[-0.01242833  0.02854099  0.00304787  0.06350479  0.00300171]


# Time Feature Engineering

In [35]:
behaviors = pd.read_pickle("processed_data/behaviors_clean.pkl")

print("Loaded behaviors:", behaviors.shape)

Loaded behaviors: (156965, 5)


In [36]:
behaviors["hour"] = behaviors["time"].dt.hour
behaviors["day_of_week"] = behaviors["time"].dt.weekday  # Monday=0

In [37]:
behaviors["hour_sin"] = np.sin(2 * np.pi * behaviors["hour"] / 24)
behaviors["hour_cos"] = np.cos(2 * np.pi * behaviors["hour"] / 24)

In [38]:
behaviors["day_sin"] = np.sin(2 * np.pi * behaviors["day_of_week"] / 7)
behaviors["day_cos"] = np.cos(2 * np.pi * behaviors["day_of_week"] / 7)

In [39]:
time_features = behaviors[[
    "hour_sin", "hour_cos",
    "day_sin", "day_cos"
]].values

print("Time feature shape:", time_features.shape)

Time feature shape: (156965, 4)


In [40]:
behaviors = behaviors.sort_values(["user_id", "time"])

behaviors["last_time"] = behaviors.groupby("user_id")["time"].shift(1)

behaviors["time_diff"] = (
    (behaviors["time"] - behaviors["last_time"])
    .dt.total_seconds()
    .fillna(0)
)

# Normalize (log scale)
behaviors["time_diff"] = np.log1p(behaviors["time_diff"])

In [41]:
time_features = behaviors[[
    "hour_sin", "hour_cos",
    "day_sin", "day_cos",
    "time_diff"
]].values

In [42]:
time_features = normalize(time_features)

In [43]:
np.save("processed_data/time_features.npy", time_features)

# Build FINAL NEWS VECTOR

In [65]:
text_embeddings = np.load("processed_data/text_embeddings.npy")
entity_embeddings = np.load("processed_data/entity_embeddings.npy")
category_embeddings = np.load("processed_data/category_embeddings.npy")


news = pd.read_pickle("processed_data/news_clean.pkl")
news_ids = news["news_id"].tolist()

print(text_embeddings.shape)
print(entity_embeddings.shape)
print(category_embeddings.shape)

(51282, 384)
(51282, 100)
(51282, 384)


In [66]:
news_vectors = np.concatenate(
    [text_embeddings, entity_embeddings, category_embeddings],
    axis=1
)

print("Final news vector shape:", news_vectors.shape)

Final news vector shape: (51282, 868)


In [67]:
news_vectors = normalize(news_vectors)

In [68]:
np.save("processed_data/news_vectors.npy", news_vectors)

In [69]:
news_vector_dict = {
    news_id: vec for news_id, vec in zip(news_ids, news_vectors)
}

with open("processed_data/news_vector_dict.pkl", "wb") as f:
    pickle.dump(news_vector_dict, f)

In [50]:
print(news_vectors.shape)
print(news_vectors[0][:10])

(51282, 1636)
[-0.00379115  0.01732703  0.02408186  0.00494211  0.01364333  0.00802023
  0.0111732  -0.02511837 -0.01476533 -0.01594105]
